# 01 · Data preparation

Builds the child-level analysis dataset from the Pakistan Social and Living Standards
Measurement Survey (PSLM) 2019-20.

**Input:** `raw_data/stata data/*.dta` (Pakistan Bureau of Statistics).
**Output:** `data/analysis_data.parquet`, `data/variables.csv`.

1. Setup
2. Study districts
3. Predictors
4. Sample
5. Outcome
6. Quality checks
7. Save

## Step 1 · Setup

In [1]:
import sys
sys.path.insert(0, "..")
import src  # noqa: F401  - loads TensorFlow before pandas (required on Windows)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

from src.build_dataset import build_dataset
from src.config import (CONSTRUCTED, DATA_DIR, DISTRICTS, FEATURE_LEVEL, FEATURES, LABEL,
                        NUMERIC_FEATURES, RAW_DIR, SCHOOL_DEPENDENT, SURVEY_ITEMS, TARGET)

DATA_DIR.mkdir(exist_ok=True)
sorted(p.name for p in RAW_DIR.glob("*.dta"))

['info.dta',
 'plist.dta',
 'roster.dta',
 'secb2.dta',
 'secc1.dta',
 'secc2.dta',
 'secd.dta',
 'sece.dta',
 'secf1.dta',
 'secf2.dta',
 'secf3.dta',
 'secg.dta',
 'sech.dta',
 'seci.dta',
 'secj.dta',
 'seck.dta',
 'secl.dta',
 'weight_file.dta']

## Step 2 · Study districts
Nine districts: the federal capital, the four provincial capitals, and one high-deprivation district per province (Multidimensional Poverty Index).

In [2]:
pd.DataFrame([(v[1], v[2], v[3], v[0]) for v in DISTRICTS.values()],
             columns=["District", "Type", "MPI", "PSLM code"])

,District,Type,MPI,PSLM code
0,Karachi Central,Federal and Provincial Capitals,0.02,307
1,Islamabad,Federal and Provincial Capitals,0.01,212
2,Lahore,Federal and Provincial Capitals,0.03,218
3,Quetta,Federal and Provincial Capitals,0.20,426
4,Peshawar,Federal and Provincial Capitals,0.11,125
5,Kohistan,High-Deprivation,0.53,114
6,Dera Bugti,High-Deprivation,0.36,404
7,Rajanpur,High-Deprivation,0.32,230
8,Tharparkar,High-Deprivation,0.51,327


## Step 3 · Predictors
56 candidate predictors: 48 survey questions used as recorded and 8 constructed variables, each measured either for the child or for the household (shared by siblings). Three are recorded only for children who attend or attended school (*school-dependent*); they are excluded from the models in notebook 03.

In [3]:
variables = pd.DataFrame(
    [(name, f"{m}.{v}", "question") for name, (m, v) in SURVEY_ITEMS.items()]
    + [(name, text, "constructed") for name, text in CONSTRUCTED.items()],
    columns=["variable", "source / definition", "kind"])
variables["level"] = variables["variable"].map(FEATURE_LEVEL)
variables["type"] = np.where(variables["variable"].isin(NUMERIC_FEATURES), "numeric", "categorical")
variables["school-dependent"] = variables["variable"].isin(SCHOOL_DEPENDENT)
variables.to_csv(DATA_DIR / "variables.csv", index=False)
variables

,variable,source / definition,kind,level,type,school-dependent
0,age,roster.age,question,child,numeric,False
1,gender,roster.sb1q4,question,child,categorical,False
2,relationship_to_head,roster.sb1q2,question,child,categorical,False
3,reason_for_headship,roster.sb1q3,question,household,categorical,False
4,residence_status,roster.sb1q5,question,child,categorical,False
5,marital_status,roster.sb1q7,question,child,categorical,False
6,region,roster.region,question,household,categorical,False
7,born_in_district,secb2.sb2q01,question,child,categorical,False
8,born_district_type,secb2.sb2q2a,question,child,categorical,False
9,migration_reason,secb2.sb2q05,question,child,categorical,False


## Step 4 · Sample
Children aged 5-16 who are usual members of a sampled household (roster `sb1q11 = 1`), with their household sampling weight and primary sampling unit (PSU).

In [4]:
analysis = build_dataset()
analysis.groupby("district_key").agg(children=("idc", "size"),
                                     households=("hhcode", "nunique"),
                                     PSUs=("psu", "nunique")).rename(index=LABEL)

,children,households,PSUs
district_key,,,
Dera Bugti,1270,455,19
Islamabad,1696,771,50
Karachi Central,2917,1462,117
Kohistan,2604,943,54
Lahore,7956,3524,225
Peshawar,6315,2037,90
Quetta,3259,1113,62
Rajanpur,2833,1012,44
Tharparkar,1742,675,36


## Step 5 · Outcome
`education_access` = 1 if the child is currently attending school (`secc1 sc1q01 = 3`), 0 otherwise.

In [5]:
outcome = analysis.groupby("district_key")[TARGET].agg(attending="sum", children="size")
outcome["out of school"] = outcome["children"] - outcome["attending"]
outcome["out of school %"] = (outcome["out of school"] / outcome["children"] * 100).round(1)
outcome.rename(index=LABEL)

,attending,children,out of school,out of school %
district_key,,,,
Dera Bugti,275,1270,995,78.3
Islamabad,1517,1696,179,10.6
Karachi Central,2281,2917,636,21.8
Kohistan,951,2604,1653,63.5
Lahore,6668,7956,1288,16.2
Peshawar,4583,6315,1732,27.4
Quetta,2125,3259,1134,34.8
Rajanpur,1453,2833,1380,48.7
Tharparkar,739,1742,1003,57.6


## Step 6 · Quality checks
One row per child, valid ages and outcome, every child weighted; status of household income; share of missing values per predictor.

In [6]:
assert not analysis.duplicated(["hhcode", "idc"]).any(), "one row per child"
assert analysis["age"].between(5, 16).all()
assert analysis[TARGET].isin([0, 1]).all()
assert analysis["weights"].notna().all(), "every child has a sampling weight"

print("Household income status (households):")
display(analysis.drop_duplicates("hhcode")["income_status"].value_counts().to_frame("households"))
missing = (analysis[FEATURES].isna().mean() * 100).round(1).sort_values(ascending=False)
print("Missing values, % of children (all districts):")
missing.to_frame("% missing")

Household income status (households):


,households
income_status,
reported,11554
unmeasured,372
zero,60
not answered,6


Missing values, % of children (all districts):


,% missing
monthly_income,98.3
can_report_income,98.2
work_days_last_month,95.5
employment_status,95.5
years_to_complete_primary,95.1
born_district_type,95.1
migration_reason,95.1
computer_location,94.3
property_owner_gender,81.1
transport_mode,69.9


## Step 7 · Save

In [7]:
analysis.to_parquet(DATA_DIR / "analysis_data.parquet", index=False)
print("saved: data/analysis_data.parquet", analysis.shape)

saved: data/analysis_data.parquet (30592, 64)
